# Basic attack 

For each respondent on the leaked list we look up their sex, age group and voting channel in the public register, find all released rows with the same profile, and guess the most common party among them.

In [1]:
import pandas as pd
from datetime import date

CUTOFF = 0.80          # only submit guesses we are at least this sure about

In [2]:
released = pd.read_csv(r"../dataC/anonymised_dataC.csv", dtype=str)
register = pd.read_excel(r"../dataC/public_data_registerC.xlsx")
names = [line.strip() for line in open(r"../dataC/survey_listC.txt", encoding="utf-8") if line.strip()]

print("released rows:", len(released), "----- register people:", len(register), "---- leaked names:", len(names))
released.head()

released rows: 200 ----- register people: 1527 ---- leaked names: 200


,sex,age_group,education,evote,party
0,Female,60+,1 Primary,0,Red
1,Male,20-39,2 Upper secondary/VET,0,Green
2,Female,40-59,3 Short/medium higher,0,Red
3,Male,20-39,2 Upper secondary/VET,0,Green
4,Male,20-39,2 Upper secondary/VET,0,Green


In [3]:
# look up the responders in the register
responders = register[register["name"].isin(names)].copy()
print("respondents found in register:", len(responders))
responders.head()

respondents found in register: 200


,name,sex,dob,zip,citizenship,marital_status,last_voted
3,"Cole, Amos",Male,1998-08-04,2300,Denmark,Never married,1
4,"Gibson, Isaiah",Male,2001-03-15,2100,Denmark,Never married,1
8,"Piester, Emma",Female,2004-01-11,2400,Denmark,Never married,1
24,"el-Fahmy, Fikri",Male,2004-01-13,2300,Denmark,Never married,1
27,"Spoon, Kelsey",Female,1999-02-20,2400,Denmark,Never married,1


#### recode them the way group C did
Age on 1 July 2025 in 3 bands (20-39, 40-59, 60+). The voting channel evote, coming from `last_voted`

In [5]:
ref = date(2025, 7, 1)
age = responders["dob"].apply(lambda d: ref.year - d.year - ((ref.month, ref.day) < (d.month, d.day)))
responders["age_group"] = pd.cut(age, [0, 40, 60, 200], right=False, labels=["20-39", "40-59", "60+"]).astype(str)
responders["evote"] = responders["last_voted"].astype(str)
profile = ["sex", "age_group", "evote"]

# sanity check
check = pd.DataFrame({"from register": responders.groupby(profile).size(), "in release": released.groupby(profile).size()})

check["from register"].sum()


np.int64(200)

## 4. Most common party per profile
Share of each party inside every profile group of the release; keep the top party per profile. The share is our confidence: the chance that a person in that group voted that party.

In [6]:
shares = released.groupby(profile)["party"].value_counts(normalize=True).rename("confidence").reset_index()
best = shares.sort_values("confidence", ascending=False).drop_duplicates(profile)
best

,sex,age_group,evote,party,confidence
14,Male,20-39,0,Green,0.900000
17,Male,20-39,1,Green,0.833333
2,Female,20-39,1,Green,0.777778
6,Female,40-59,1,Green,0.714286
0,Female,20-39,0,Green,0.705882
19,Male,40-59,0,Green,0.700000
8,Female,60+,0,Red,0.650000
11,Female,60+,1,Red,0.600000
24,Male,60+,0,Green,0.600000
4,Female,40-59,0,Green,0.588235


## 5. Give each respondent the best guess of their profile

In [9]:
resp = responders.merge(best, on=profile, how="left")
print("expected accuracy over all 200:", round(resp["confidence"].mean(), 3))
resp[["name", "sex", "age_group", "evote", "party", "confidence"]].sort_values("confidence", ascending=False).head(10)

expected accuracy over all 200: 0.675


,name,sex,age_group,evote,party,confidence
64,"Cordiner, Darren",Male,20-39,0,Green,0.9
67,"Ransom, Joshua",Male,20-39,0,Green,0.9
59,"el-Mannan, Rajaa",Male,20-39,0,Green,0.9
57,"Soukup, Tyler",Male,20-39,0,Green,0.9
60,"Watkins, Ryan",Male,20-39,0,Green,0.9
61,"Bittenbinder, Paul",Male,20-39,0,Green,0.9
68,"Rinhart, Brendan",Male,20-39,0,Green,0.9
71,"Cheatham, Tevin",Male,20-39,0,Green,0.9
62,"Ontiveros, Carlos",Male,20-39,0,Green,0.9
63,"Wang, Vinh",Male,20-39,0,Green,0.9


## 6. Write the confident guesses as `Name; party`

In [10]:
sure = resp[resp["confidence"] >= CUTOFF].sort_values("confidence", ascending=False)
with open("basic_deanonymised_C.csv", "w", encoding="utf-8", newline="") as f:
    for name, party in zip(sure["name"], sure["party"]):
        f.write(f"{name}; {party.lower()}\n")

print(f"{len(sure)} voters with confidence >= {CUTOFF}, expected correct {sure['confidence'].sum():.1f}")

32 voters with confidence >= 0.8, expected correct 28.0
